# SFT vs GRPO: Stream-of-Consciousness Detection — Comparative Analysis

This notebook loads evaluation results from `outputs/notebook_demo` and compares two fine-tuning strategies applied to **Qwen3-4B**:

- **v3_sft** — Supervised Fine-Tuning (SFT)
- **v3_grpo** — Group Relative Policy Optimization (GRPO, a reinforcement-learning approach)

The task is binary detection of *stream-of-consciousness* (SoC) passages in literary text, plus fine-grained type classification.


## 1. Load and Inspect Data

In [ ]:
import json
import pathlib
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 30)

# ── locate the demo output directory ──────────────────────────────────────────
BASE = pathlib.Path("outputs/notebook_demo")
files = sorted(BASE.glob("*.csv")), sorted(BASE.glob("*.json"))

# ── load every artefact ───────────────────────────────────────────────────────
manifest   = json.loads((BASE / next(BASE.glob("*_manifest.json")).name).read_text())
summary    = pd.read_csv(next(BASE.glob("*_summary.csv")))
preds      = pd.read_csv(next(BASE.glob("*_predictions.csv")))
compare    = pd.read_csv(next(BASE.glob("*_compare.csv")))
risk       = pd.read_csv(next(BASE.glob("*_risk_curves.csv")))

MODELS = manifest["models"]          # ["v3_sft", "v3_grpo"]
PALETTE = {"v3_sft": "#1f77b4", "v3_grpo": "#ff7f0e"}
LABELS  = {"v3_sft": "SFT", "v3_grpo": "GRPO"}

print("Run tag :", manifest["run_tag"])
print("Base model (short):", manifest["base_model"].split("/")[-3])
print("Models evaluated  :", MODELS)
print(f"\nPrediction rows   : {len(preds)}  ({len(preds)//len(MODELS)} cases × {len(MODELS)} models)")
print(f"Compare rows      : {len(compare)} cases")
print(f"Risk-curve rows   : {len(risk)}")
summary.set_index("model_id")


In [ ]:
# Quick peek at the predictions table (one row per model × case)
preds[["model_id","id","kind","gold_is_soc","pred_is_soc","pred_type",
       "pred_confidence","pred_confidence_num","correct_binary"]].head(10)


## 2. Aggregate Metrics by Training Method

Key metrics:
| Metric | Description |
|---|---|
| **Precision** | Of all SoC predictions, fraction correct — measures *false-positive rate* |
| **Recall** | Of all true SoC passages, fraction caught — measures *coverage* |
| **F1** | Harmonic mean of precision and recall |
| **Accuracy** | Overall binary correctness |
| **ECE proxy** | Expected Calibration Error proxy — lower = confidence better matches accuracy |
| **Brier proxy** | Probability scoring rule — lower = sharper, better-calibrated probabilities |
| **AURC proxy** | Area Under the Risk-Coverage curve — lower = safer selective prediction |


In [ ]:
# ── summary table ─────────────────────────────────────────────────────────────
display_summary = summary.copy()
display_summary["model"] = display_summary["model_id"].map(LABELS)
display_summary = display_summary.set_index("model").drop(columns=["model_id"])
display_summary.index.name = None

# highlight better value in each column
def highlight_better(col):
    # lower is better for error metrics, higher for the rest
    lower_is_better = col.name in ("ece_proxy", "brier_proxy", "aurc_proxy", "fn", "fp")
    best = col.min() if lower_is_better else col.max()
    return ["font-weight: bold; background-color: #d4f0d4" if v == best else "" for v in col]

display_summary.style.apply(highlight_better).format(precision=4)


In [ ]:
# ── bar chart: key performance metrics ────────────────────────────────────────
perf_metrics = ["precision", "recall", "f1", "accuracy"]
error_metrics = ["ece_proxy", "brier_proxy", "aurc_proxy"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# performance (higher = better)
ax = axes[0]
x = np.arange(len(perf_metrics))
w = 0.35
for i, mid in enumerate(MODELS):
    row = summary[summary["model_id"] == mid].iloc[0]
    vals = [row[m] for m in perf_metrics]
    bars = ax.bar(x + i * w, vals, w, label=LABELS[mid], color=PALETTE[mid], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x + w / 2)
ax.set_xticklabels([m.replace("_", "\n") for m in perf_metrics])
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Performance metrics (higher = better)")
ax.legend()
ax.axhline(1.0, color="grey", lw=0.5, ls="--")

# error / calibration (lower = better)
ax = axes[1]
x = np.arange(len(error_metrics))
for i, mid in enumerate(MODELS):
    row = summary[summary["model_id"] == mid].iloc[0]
    vals = [row[m] for m in error_metrics]
    bars = ax.bar(x + i * w, vals, w, label=LABELS[mid], color=PALETTE[mid], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x + w / 2)
ax.set_xticklabels([m.replace("_", "\n") for m in error_metrics])
ax.set_ylabel("Score")
ax.set_title("Calibration / risk metrics (lower = better)")
ax.legend()

plt.suptitle("SFT vs GRPO — Aggregate Evaluation", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 3. Confidence Score Distributions

Each prediction carries a numeric confidence (`pred_confidence_num`). We examine how the two models distribute their certainty, and whether high confidence actually maps to correct predictions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, mid in zip(axes, MODELS):
    df_m = preds[preds["model_id"] == mid].copy()
    correct   = df_m[df_m["correct_binary"] == True]["pred_confidence_num"]
    incorrect = df_m[df_m["correct_binary"] == False]["pred_confidence_num"]

    bins = np.linspace(0.5, 1.0, 8)
    ax.hist(correct,   bins=bins, alpha=0.7, color="steelblue", label="Correct",   edgecolor="white")
    ax.hist(incorrect, bins=bins, alpha=0.7, color="tomato",    label="Incorrect", edgecolor="white")

    mean_c = correct.mean()   if len(correct)   else float("nan")
    mean_i = incorrect.mean() if len(incorrect) else float("nan")
    if not np.isnan(mean_c):
        ax.axvline(mean_c,   color="steelblue", ls="--", lw=1.5, label=f"Mean correct ({mean_c:.2f})")
    if not np.isnan(mean_i):
        ax.axvline(mean_i,   color="tomato",    ls="--", lw=1.5, label=f"Mean incorrect ({mean_i:.2f})")

    ax.set_title(f"{LABELS[mid]}  —  confidence distribution", fontweight="bold")
    ax.set_xlabel("Confidence score")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.set_xlim(0.55, 1.05)

plt.suptitle("Do higher-confidence predictions correlate with correctness?", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# ── numerical summary ──────────────────────────────────────────────────────────
print("Confidence summary by model and correctness:\n")
conf_summary = (
    preds.groupby(["model_id", "correct_binary"])["pred_confidence_num"]
    .agg(["count", "mean", "std", "min", "max"])
    .rename(columns={"count": "n"})
)
conf_summary.index = conf_summary.index.set_levels(
    [LABELS.get(v, v) for v in conf_summary.index.levels[0]], level=0
)
conf_summary


## 4. Win / Loss / Tie Analysis

For each of the 14 test cases we ask: did SFT get it right? Did GRPO? We classify every case into one of four outcomes and also show a **confusion-style 2×2 matrix** of joint correctness.


In [ ]:
# ── build per-case correct flags from compare table ───────────────────────────
# compare has one row per case; derive correct_binary for each model
compare2 = compare.copy()
compare2["correct_grpo"] = compare2["pred_is_soc_v3_grpo"] == compare2["gold_is_soc"]
compare2["correct_sft"]  = compare2["pred_is_soc_v3_sft"]  == compare2["gold_is_soc"]

def outcome(row):
    if row["correct_sft"] and row["correct_grpo"]:
        return "Both correct"
    elif row["correct_sft"] and not row["correct_grpo"]:
        return "SFT wins"
    elif not row["correct_sft"] and row["correct_grpo"]:
        return "GRPO wins"
    else:
        return "Both wrong"

compare2["outcome"] = compare2.apply(outcome, axis=1)
outcome_counts = compare2["outcome"].value_counts().reindex(
    ["Both correct", "SFT wins", "GRPO wins", "Both wrong"], fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# stacked bar
ax = axes[0]
colors = {"Both correct": "#2ca02c", "SFT wins": "#1f77b4",
          "GRPO wins": "#ff7f0e", "Both wrong": "#d62728"}
bottom = 0
for label, cnt in outcome_counts.items():
    ax.bar("All cases", cnt, bottom=bottom, color=colors[label], label=label, width=0.4)
    if cnt > 0:
        ax.text(0, bottom + cnt / 2, f"{label}\n({cnt})",
                ha="center", va="center", fontsize=9, color="white", fontweight="bold")
    bottom += cnt
ax.set_ylabel("Number of cases")
ax.set_title("Win / Loss / Tie breakdown")
ax.legend(loc="upper right", fontsize=8)
ax.set_ylim(0, len(compare2) + 1)

# 2×2 joint-correctness heatmap
ax = axes[1]
tbl = pd.crosstab(compare2["correct_sft"].map({True: "SFT correct", False: "SFT wrong"}),
                  compare2["correct_grpo"].map({True: "GRPO correct", False: "GRPO wrong"}))
tbl = tbl.reindex(index=["SFT correct", "SFT wrong"],
                  columns=["GRPO correct", "GRPO wrong"], fill_value=0)
im = ax.imshow(tbl.values, cmap="Blues", aspect="auto", vmin=0)
ax.set_xticks([0, 1]); ax.set_xticklabels(tbl.columns, fontsize=10)
ax.set_yticks([0, 1]); ax.set_yticklabels(tbl.index, fontsize=10)
for (r, c), v in np.ndenumerate(tbl.values):
    ax.text(c, r, str(v), ha="center", va="center", fontsize=16, fontweight="bold",
            color="white" if v > 2 else "black")
ax.set_title("Joint correctness (cases)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("SFT vs GRPO — Per-Case Outcome", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("\nOutcome counts:")
print(outcome_counts.to_string())


## 5. Per-Category Performance Breakdown

Cases are tagged with a `kind` label (`clear_soc`, `ambiguous`, `real_soc`, `clear_non_soc`, `real_non_soc`). We examine accuracy and confidence for each model across these categories to reveal systematic strengths and gaps.


In [ ]:
cat_acc = (
    preds.groupby(["kind", "model_id"])
    .agg(
        accuracy=("correct_binary", "mean"),
        confidence=("pred_confidence_num", "mean"),
        n=("correct_binary", "count"),
    )
    .reset_index()
)
cat_acc["model_label"] = cat_acc["model_id"].map(LABELS)

categories = sorted(cat_acc["kind"].unique())
x = np.arange(len(categories))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title, ylbl in [
    (axes[0], "accuracy",   "Accuracy per category",   "Accuracy"),
    (axes[1], "confidence", "Mean confidence per category", "Confidence"),
]:
    for i, mid in enumerate(MODELS):
        sub = cat_acc[cat_acc["model_id"] == mid].set_index("kind")
        vals = [sub.loc[c, metric] if c in sub.index else 0 for c in categories]
        bars = ax.bar(x + i * w, vals, w, label=LABELS[mid], color=PALETTE[mid], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(x + w / 2)
    ax.set_xticklabels([c.replace("_", "\n") for c in categories], fontsize=9)
    ax.set_ylim(0, 1.2)
    ax.set_ylabel(ylbl)
    ax.set_title(title, fontweight="bold")
    ax.legend()
    ax.axhline(1.0, color="grey", lw=0.5, ls="--")

plt.suptitle("SFT vs GRPO — Category-Level Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# table
pivot_acc = cat_acc.pivot_table(
    index="kind", columns="model_label", values="accuracy", aggfunc="mean"
).round(3)
pivot_acc.columns.name = None
pivot_acc.index.name = "Category"
pivot_acc["Δ (SFT − GRPO)"] = pivot_acc.get("SFT", 0) - pivot_acc.get("GRPO", 0)
pivot_acc


In [ ]:
# ── risk-coverage curves ──────────────────────────────────────────────────────
# Lower risk at a given coverage level = better selective prediction
fig, ax = plt.subplots(figsize=(8, 5))
for mid in MODELS:
    r = risk[risk["model_id"] == mid]
    ax.plot(r["coverage"], r["risk"], marker="o", markersize=4,
            label=LABELS[mid], color=PALETTE[mid], linewidth=2)

ax.set_xlabel("Coverage (fraction of cases model answers)", fontsize=11)
ax.set_ylabel("Risk (error rate among answered cases)", fontsize=11)
ax.set_title("Risk-Coverage Curve\n(lower-left is better — more coverage at lower risk)",
             fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(0, 1.05); ax.set_ylim(-0.02, 0.65)
ax.fill_between(risk[risk["model_id"]=="v3_sft"]["coverage"],
                risk[risk["model_id"]=="v3_sft"]["risk"],
                alpha=0.1, color=PALETTE["v3_sft"])
ax.fill_between(risk[risk["model_id"]=="v3_grpo"]["coverage"],
                risk[risk["model_id"]=="v3_grpo"]["risk"],
                alpha=0.1, color=PALETTE["v3_grpo"])

# annotate AURC values
for mid in MODELS:
    aurc = summary[summary["model_id"]==mid]["aurc_proxy"].values[0]
    ax.annotate(f"AURC = {aurc:.4f}", xy=(1.01, risk[risk["model_id"]==mid]["risk"].values[-1]),
                fontsize=9, color=PALETTE[mid])

plt.tight_layout()
plt.show()


## 6. Error and Failure Mode Analysis

We look at *what went wrong* for each model: which cases were missed, whether both models share the same blind spots, and how confidence behaves on failures.


In [ ]:
# ── false-negative analysis (missed SoC passages) ─────────────────────────────
# Both models have perfect precision (no FP), so all errors are false negatives
fn_grpo = compare2[~compare2["correct_grpo"] & compare2["gold_is_soc"]][["id","kind","gold_primary","pred_is_soc_v3_grpo","pred_type_v3_grpo"]].copy()
fn_sft  = compare2[~compare2["correct_sft"]  & compare2["gold_is_soc"]][["id","kind","gold_primary","pred_is_soc_v3_sft","pred_type_v3_sft"]].copy()
fn_both = compare2[~compare2["correct_grpo"] & ~compare2["correct_sft"] & compare2["gold_is_soc"]][["id","kind","gold_primary"]]

print("=== False Negatives (missed SoC) ===")
print(f"\nGRPO missed {len(fn_grpo)} SoC cases:")
print(fn_grpo.to_string(index=False))
print(f"\nSFT missed {len(fn_sft)} SoC cases:")
print(fn_sft.to_string(index=False))
print(f"\nBoth missed (shared blind spots) — {len(fn_both)} cases:")
print(fn_both.to_string(index=False))


In [ ]:
# ── error pattern visualisation ───────────────────────────────────────────────
error_breakdown = pd.DataFrame({
    "Category": ["Shared FN\n(both wrong)", "GRPO-only FN\n(SFT correct)", "SFT-only FN\n(GRPO correct)"],
    "Count":    [len(fn_both),
                 len(fn_grpo) - len(fn_both),
                 len(fn_sft)  - len(fn_both)],
    "Color":    ["#d62728", "#ff7f0e", "#1f77b4"],
})

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
bars = ax.bar(error_breakdown["Category"], error_breakdown["Count"],
              color=error_breakdown["Color"], width=0.5, alpha=0.85, edgecolor="white")
for bar, v in zip(bars, error_breakdown["Count"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(v), ha="center", va="bottom", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of false negatives")
ax.set_title("Where each model uniquely fails", fontweight="bold")
ax.set_ylim(0, max(error_breakdown["Count"]) + 1)

# confidence on errors vs correct per model
ax = axes[1]
conf_err = []
for mid, label in LABELS.items():
    df_m = preds[preds["model_id"] == mid]
    err  = df_m[df_m["correct_binary"] == False]["pred_confidence_num"].dropna()
    ok   = df_m[df_m["correct_binary"] == True]["pred_confidence_num"].dropna()
    conf_err.append({"Model": label, "Group": "Errors",   "mean_conf": err.mean() if len(err) else 0})
    conf_err.append({"Model": label, "Group": "Correct",  "mean_conf": ok.mean()  if len(ok)  else 0})

conf_df = pd.DataFrame(conf_err)
groups = conf_df["Group"].unique()
x = np.arange(len(groups))
w = 0.35
for i, (mid, label) in enumerate(LABELS.items()):
    sub = conf_df[conf_df["Model"] == label].set_index("Group")
    vals = [sub.loc[g, "mean_conf"] if g in sub.index else 0 for g in groups]
    bars = ax.bar(x + i*w, vals, w, label=label, color=PALETTE[mid], alpha=0.85)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x + w/2); ax.set_xticklabels(groups)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Mean confidence score")
ax.set_title("Mean confidence: errors vs correct predictions", fontweight="bold")
ax.legend()

plt.suptitle("Failure Mode Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 7. Side-by-Side Response Quality Examples

The most informative comparison is qualitative: cases where SFT and GRPO *disagree*. These show the models' different decision boundaries — SFT tends to be more sensitive (higher recall), GRPO more conservative (higher precision per case, but lower recall).


In [ ]:
from IPython.display import HTML

# cases where the two models give different binary predictions
disagreements = compare2[
    compare2["pred_is_soc_v3_grpo"] != compare2["pred_is_soc_v3_sft"]
].copy()
disagreements["sft_correct"]  = disagreements["correct_sft"]
disagreements["grpo_correct"] = disagreements["correct_grpo"]

# pull in prediction details from preds table
def get_pred_details(case_id, model_id):
    row = preds[(preds["id"] == case_id) & (preds["model_id"] == model_id)]
    if row.empty:
        return {}
    r = row.iloc[0]
    return {
        "pred_is_soc":    r["pred_is_soc"],
        "pred_type":      r.get("pred_type", "—"),
        "pred_secondary": r.get("pred_secondary", ""),
        "confidence":     r.get("pred_confidence", ""),
        "explanation":    str(r.get("pred_explanation", ""))[:400],
        "think_preview":  str(r.get("think_preview", ""))[:300],
    }

rows_html = []
for _, row in disagreements.iterrows():
    sft  = get_pred_details(row["id"], "v3_sft")
    grpo = get_pred_details(row["id"], "v3_grpo")
    gold_label = "✅ SoC" if row["gold_is_soc"] else "❌ Not SoC"
    sft_check  = "✅" if row["sft_correct"]  else "❌"
    grpo_check = "✅" if row["grpo_correct"] else "❌"

    rows_html.append(f"""
<tr style="border-top:2px solid #ccc;">
  <td colspan="2" style="background:#f5f5f5;padding:6px 10px;font-weight:bold;">
    Case: <code>{row['id']}</code> &nbsp;|&nbsp;
    Kind: <em>{row['kind']}</em> &nbsp;|&nbsp;
    Gold primary: <em>{row.get('gold_primary','?')}</em> &nbsp;|&nbsp;
    Gold label: <strong>{gold_label}</strong>
  </td>
</tr>
<tr>
  <td style="width:50%;vertical-align:top;padding:8px 12px;border-right:1px solid #ddd;">
    <b style="color:#1f77b4;">SFT {sft_check}</b><br>
    <b>Prediction:</b> {'SoC' if sft.get('pred_is_soc') else 'Not SoC'}
      (type: {sft.get('pred_type','—')}, conf: {sft.get('confidence','—')})<br>
    <b>Secondary signals:</b> {sft.get('pred_secondary','—') or '—'}<br><br>
    <b>Explanation:</b><br><span style="font-size:0.9em">{sft.get('explanation','—')}</span><br><br>
    <b>Thinking preview:</b><br>
    <span style="font-size:0.85em;color:#555">{sft.get('think_preview','—')}</span>
  </td>
  <td style="width:50%;vertical-align:top;padding:8px 12px;">
    <b style="color:#ff7f0e;">GRPO {grpo_check}</b><br>
    <b>Prediction:</b> {'SoC' if grpo.get('pred_is_soc') else 'Not SoC'}
      (type: {grpo.get('pred_type','—')}, conf: {grpo.get('confidence','—')})<br>
    <b>Secondary signals:</b> {grpo.get('pred_secondary','—') or '—'}<br><br>
    <b>Explanation:</b><br><span style="font-size:0.9em">{grpo.get('explanation','—')}</span><br><br>
    <b>Thinking preview:</b><br>
    <span style="font-size:0.85em;color:#555">{grpo.get('think_preview','—')}</span>
  </td>
</tr>
""")

table_html = f"""
<style>
  .compare-table {{ border-collapse:collapse; width:100%; font-family:sans-serif; font-size:0.92em; }}
  .compare-table td {{ border:1px solid #e0e0e0; }}
</style>
<table class="compare-table">
  <thead>
    <tr style="background:#333;color:white;">
      <th style="width:50%;padding:8px;">SFT (v3_sft)</th>
      <th style="width:50%;padding:8px;">GRPO (v3_grpo)</th>
    </tr>
  </thead>
  <tbody>
    {''.join(rows_html)}
  </tbody>
</table>
"""
HTML(table_html)


## Summary of Findings

| Dimension | SFT (`v3_sft`) | GRPO (`v3_grpo`) |
|---|---|---|
| **Recall** | 0.556 — catches more SoC | 0.333 — misses more SoC |
| **Precision** | 1.0 — no false positives | 1.0 — no false positives |
| **F1 / Accuracy** | 0.714 / 71.4% ✅ | 0.500 / 57.1% |
| **Calibration (ECE)** | 0.096 — better calibrated ✅ | 0.175 — overconfident on misses |
| **Brier score** | 0.168 ✅ | 0.227 |
| **Risk at full coverage** | 0.286 ✅ | 0.429 |
| **Shared failure cases** | `ambig_indirect_omniscient_1`, `real_02`, `soc_indirect_1` — passages where the SoC signal is syntactically embedded / ambiguous |

### Key takeaways

1. **SFT is strictly better on this benchmark.** It has higher recall, better calibration, and a substantially lower AURC — meaning it remains safer even when forced to answer every case.

2. **Both models achieve perfect precision.** Neither makes false positive errors (flagging non-SoC text as SoC). This suggests the RL objective in GRPO successfully learned the penalty for over-labelling — but at the cost of recall.

3. **GRPO is conservative to a fault.** Its lower recall stems from defaulting to "not SoC" on ambiguous and real-world passages. This is consistent with GRPO learning a *cautious* policy: it only fires when highly confident, which avoids false positives but inflates false negatives.

4. **SFT is more sensitive to subtle SoC signals.** The three cases SFT gets right that GRPO misses (`ambig_hybrid_1`, `real_00`, `soc_orthographic_1`) involve stylistic markers (orthographic cues, imagery, hybrid forms) rather than prototypical stream-of-consciousness patterns. SFT's exposure to labelled positives in training appears to have instilled broader pattern recognition.

5. **Shared blind spots point to genuinely hard cases.** The three cases both models miss (`ambig_indirect_omniscient_1`, `real_02`, `soc_indirect_1`) all involve indirect or embedded SoC, suggesting a data-quality / annotation challenge rather than a model-specific weakness.

6. **Risk-coverage curves confirm SFT's safety advantage.** SFT's curve rises later and peaks lower, meaning even at high abstention thresholds SFT makes fewer mistakes than GRPO at the same coverage level.
